In [1]:
%pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from torchvision.datasets import MNIST

In [3]:
#Soft max este functia de activare
#transforma scorul normal (logits) intr-o distr de probabilitate (intre 0,1 si suma all e 1) 
#ex: [2.0,1.0,0.1] -> [0.65,0.24,0.11]
def softMax(Z):
    goodZ= Z - np.max(Z,axis=1,keepdims=True)
    expZ= np.exp(goodZ)
    return expZ/np.sum(expZ,axis=1,keepdims=True)

def one_hot(y,num_classes):
    return np.eye(num_classes)[y] 
#matricea identitate dar mut 1 pe pozitia y

def accuracy(y_true, y_pred):
    return np.mean(np.argmax(y_true, axis=1) == np.argmax(y_pred, axis=1))
#compara eticheta reala cu cea prezisa in forma onehot

def iterate_batches(X, y, batch_size,shuffle=True):
    n= X.shape[0]
    indices= np.arange(n)
    if shuffle:
        np.random.shuffle(indices)
    for i in range(0, n, batch_size):
        yield X[indices[i:i+batch_size]], y[indices[i:i+batch_size]]
#dau pe rand batchuri de date cu etichetele corespunzatoare

def forward(X, W, b):
    Z = X @ W + b
    A = softMax(Z)
    return A

def cross_entropy(y_true, y_pred):
    return -np.mean(np.sum(y_true * np.log(y_pred + 1e-12), axis=1))
#1e-12 de modificat daca e cazu

def backpropagation(X, y_true, y_pred):
    m = X.shape[0]
    dZ = y_pred - y_true
    dW = (X.T @ dZ) / m
    db = np.sum(dZ, axis=0) / m
    return dW, db

def update(W, b, dW, db, learning_rate):
    W -= learning_rate * dW
    b -= learning_rate * db
    return W, b

def train(X_train, y_train_oh, X_val, y_val_oh,
          num_classes=10, epochs=50, batch_size=100, learning_rate=0.1, seed=42):

    #initializari
    np.random.seed(seed)
    n_features = X_train.shape[1]
    W = (np.random.randn(n_features, num_classes).astype(np.float32)) * 0.01
    b = np.zeros((num_classes,), dtype=np.float32)

    history = {"train_loss": [], "train_acc": [], "val_acc": []}

    # acuratete initiala
    y_pred_init = forward(X_train, W, b)
    init_acc = accuracy(y_train_oh, y_pred_init)
    print(f"Initial train accuracy (untrained): {init_acc*100:.2f}%")

    for epoch in range(1, epochs + 1):
        # ---- training pe minibatch-uri ----
        for Xb, yb in iterate_batches(X_train, y_train_oh, batch_size, shuffle=True):
            y_pred = forward(Xb, W, b) #aplic softmax
            loss = cross_entropy(yb, y_pred) # e bine sau rau?
            dW, db = backpropagation(Xb, yb, y_pred) #unde ma duc 
            W, b = update(W, b, dW, db, learning_rate) #ma duc

        # ---- metrici  ----
        y_pred_train = forward(X_train, W, b)
        train_loss = cross_entropy(y_train_oh, y_pred_train)
        train_acc = accuracy(y_train_oh, y_pred_train)

        y_pred_val = forward(X_val, W, b)
        val_acc = accuracy(y_val_oh, y_pred_val)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch:3d} | loss={train_loss:.4f} | train_acc={train_acc*100:5.2f}% | val_acc={val_acc*100:5.2f}%")

    return W, b, history

In [4]:
def load(is_train=True):
    from torchvision.datasets import MNIST
    data = MNIST(root="./data", train=is_train, download=True,
                 transform=lambda x: np.array(x, dtype=np.float32).flatten()/255.0)
    X = np.stack([img for img, _ in data], axis=0)
    y = np.array([lbl for _, lbl in data], dtype=np.int64)
    return X, y

In [5]:
X_train, y_train = load(is_train=True)
X_test,  y_test  = load(is_train=False)
num_classes = 10
y_train_oh = one_hot(y_train, num_classes)
y_test_oh  = one_hot(y_test,  num_classes)

W, b, hist = train(
        X_train=X_train, y_train_oh=y_train_oh,
        X_val=X_test,   y_val_oh=y_test_oh,
        num_classes=num_classes,
        epochs=50,            # crește la 100–200 
        batch_size=100,       # cerința temei
        learning_rate=0.1,    # bun pentru softmax regression pe MNIST
        seed=42,
    )

y_pred_test = forward(X_test, W, b)
test_acc = accuracy(y_test_oh, y_pred_test)
print(f"\nTest accuracy: {test_acc*100:.2f}%  (ținta: ≥90%)")

Initial train accuracy (untrained): 8.03%
Epoch   1 | loss=0.3805 | train_acc=89.64% | val_acc=90.48%
Epoch   2 | loss=0.3383 | train_acc=90.61% | val_acc=91.19%
Epoch   3 | loss=0.3207 | train_acc=91.02% | val_acc=91.54%
Epoch   4 | loss=0.3083 | train_acc=91.42% | val_acc=91.78%
Epoch   5 | loss=0.3022 | train_acc=91.54% | val_acc=91.82%
Epoch   6 | loss=0.2949 | train_acc=91.69% | val_acc=91.92%
Epoch   7 | loss=0.2909 | train_acc=91.91% | val_acc=92.04%
Epoch   8 | loss=0.2862 | train_acc=92.05% | val_acc=92.15%
Epoch   9 | loss=0.2834 | train_acc=92.12% | val_acc=92.13%
Epoch  10 | loss=0.2810 | train_acc=92.18% | val_acc=92.14%
Epoch  11 | loss=0.2776 | train_acc=92.27% | val_acc=92.13%
Epoch  12 | loss=0.2762 | train_acc=92.24% | val_acc=92.22%
Epoch  13 | loss=0.2759 | train_acc=92.44% | val_acc=92.24%
Epoch  14 | loss=0.2725 | train_acc=92.41% | val_acc=92.23%
Epoch  15 | loss=0.2721 | train_acc=92.46% | val_acc=92.37%
Epoch  16 | loss=0.2700 | train_acc=92.55% | val_acc=92.29